In [9]:
import torch
import torch.nn as nn
import pandas as pd
import torch
from tqdm import tqdm
from sklearn.preprocessing import MultiLabelBinarizer
import ast

#Since cross attention is directional -> drug1 attends to drug2 ≠ drug2 attends to drug1
#So we do bi-directional cross attention and then combine the outputs together (we're using concat for that)

In [10]:
#pair fusion model
class PairFusionCrossAttention(nn.Module):
    def __init__(self, embed_dim=256, num_heads=4):
        super().__init__()

        self.cross_attn_1 = nn.MultiheadAttention(embed_dim, num_heads, batch_first=True)
        self.cross_attn_2 = nn.MultiheadAttention(embed_dim, num_heads, batch_first=True)

        self.layer_norm1 = nn.LayerNorm(embed_dim)
        self.layer_norm2 = nn.LayerNorm(embed_dim)

        self.ffn = nn.Sequential(
            nn.Linear(embed_dim * 2, embed_dim),
            nn.ReLU(),
            nn.Linear(embed_dim, embed_dim)
        )

    def forward(self, e1, e2):
        e1 = e1.unsqueeze(1)
        e2 = e2.unsqueeze(1)

        out1, _ = self.cross_attn_1(e1, e2, e2)
        out2, _ = self.cross_attn_2(e2, e1, e1)

        out1 = self.layer_norm1(e1 + out1)
        out2 = self.layer_norm2(e2 + out2)

        out1 = out1.squeeze(1)
        out2 = out2.squeeze(1)

        combined = torch.cat([out1, out2], dim=-1)
        e_pair = self.ffn(combined)

        return e_pair

In [11]:
#full pipeline 
# Load fused embeddings (SMILES → embedding)
fused_dict = torch.load("drug_fused_embeddings.pt")

# Load dataset
df = pd.read_csv("final_processed_data.csv")

# Convert string list → actual list
df["Side Effect Name"] = df["Side Effect Name"].apply(ast.literal_eval)

# Build labels
mlb = MultiLabelBinarizer()
labels = mlb.fit_transform(df["Side Effect Name"])

# Initialize model
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
pair_model = PairFusionCrossAttention().to(device)
pair_model.eval()

pair_embeddings = []
valid_labels = []

print(f"Using device: {device}")
print(f"Total rows: {len(df)}")

for i in tqdm(range(len(df)), desc="Processing pairs"):
    
    s1 = df.iloc[i]["SMILES1"]
    s2 = df.iloc[i]["SMILES2"]

    if s1 not in fused_dict or s2 not in fused_dict:
        continue

    e1 = fused_dict[s1].unsqueeze(0).float().to(device)
    e2 = fused_dict[s2].unsqueeze(0).float().to(device)

    with torch.no_grad():
        e_pair = pair_model(e1, e2)

    pair_embeddings.append(e_pair.squeeze(0).cpu())
    valid_labels.append(labels[i])

# Convert to tensors
X = torch.stack(pair_embeddings)
y = torch.tensor(valid_labels)

Using device: cpu
Total rows: 63304


Processing pairs: 100%|██████████| 63304/63304 [01:22<00:00, 766.04it/s] 
C:\Users\Diya Budlakoti\AppData\Local\Temp\ipykernel_11592\206365653.py:45: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\torch\csrc\utils\tensor_new.cpp:256.)
  y = torch.tensor(valid_labels)


In [12]:
torch.save({
    "X": X,
    "y": y,
    "side_effect_classes": mlb.classes_
}, "pair_embeddings.pt")

print("Done!")
print("Final shape:", X.shape, y.shape)

Done!
Final shape: torch.Size([63304, 256]) torch.Size([63304, 963])
